Hi! I am a Kaggle beginner and not fluent in English, so this notebook is written in Japanese.

Please use your browser's translation (or DeepL/ChatGPT) to read it!

Hope it helps!

# これまでの進捗（簡易版）

| Ver. | 内容 | LB | CV Accuracy | CV Log Loss |
| ---- | ---- | ---- | ---- | ---- |
| 1 | LightGBM によるパイプライン構築 | 0.79611 | 0.7991 | 0.4300 |
| 2 | EDA - Cabinの加工 | 0.80032 | 0.8040 | 0.4060 |
| 3 | 欠損値補完のパイプライン構築 | 0.80126 | 0.8062 | 0.4013 |
| 6 | EDA - Age関連 | 0.79962 | 0.8066 | 0.4026 |
| 7 | ハイパーパラメータの調整 | 0.79869 | 0.8094 | 0.3994 |
| 8 | 最適閾値の調査 | 0.79845 | 0.8098 | 0.3994 |
| 9 | アンサンブル - LightGBM, CatBoost | 0.80032 | 0.8109 | - |

In [404]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, log_loss
import matplotlib.pyplot as plt
import seaborn as sns

In [405]:
import kagglehub

path = kagglehub.competition_download('spaceship-titanic')

print("Path to competition files:", path)

Path to competition files: /kaggle/input/competitions/spaceship-titanic


In [406]:
train = pd.read_csv("../input/competitions/spaceship-titanic/train.csv")
test = pd.read_csv("../input/competitions/spaceship-titanic/test.csv")
sample_submission = pd.read_csv("../input/competitions/spaceship-titanic/sample_submission.csv")

# データの概要

今回の目的変数は `Transported`。　各乗客が別の次元に転送されたかどうかを予測する。

train は 8693 件、test は 4277 件。

In [407]:
train.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [408]:
test.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name
0,0013_01,Earth,True,G/3/S,TRAPPIST-1e,27.0,False,0.0,0.0,0.0,0.0,0.0,Nelly Carsoning
1,0018_01,Earth,False,F/4/S,TRAPPIST-1e,19.0,False,0.0,9.0,0.0,2823.0,0.0,Lerome Peckers
2,0019_01,Europa,True,C/0/S,55 Cancri e,31.0,False,0.0,0.0,0.0,0.0,0.0,Sabih Unhearfus
3,0021_01,Europa,False,C/1/S,TRAPPIST-1e,38.0,False,0.0,6652.0,0.0,181.0,585.0,Meratz Caltilter
4,0023_01,Earth,False,F/5/S,TRAPPIST-1e,20.0,False,10.0,0.0,635.0,0.0,0.0,Brence Harperez


In [409]:
sample_submission.head()

,PassengerId,Transported
0,0013_01,False
1,0018_01,False
2,0019_01,False
3,0021_01,False
4,0023_01,False


In [410]:
print(f"train の件数 = {len(train)}")
print(f"test の件数 = {len(test)}")
print(f"sample_submission の件数 = {len(sample_submission)}")

train の件数 = 8693
test の件数 = 4277
sample_submission の件数 = 4277


## 各列の意味（参考 : Google翻訳）

全体を通して欠損値は少ないので、全て補完して対応する方針。

- `PassengerId` : 各乗客に固有のIDが割り当てられる。各IDは、乗客が一緒に旅行しているグループを示し、グループ内での乗客の番号を示す gggg_pp 形式をとる。グループのメンバーは家族であることが多いが、必ずしもそうとは限らない。
- `HomePlanet` : 乗客が出発した惑星。通常は、その乗客の永住地である惑星。
- `CryoSleep` : 乗客が航海中、仮死状態に入ることを選択したかどうか。冷凍睡眠中の乗客は客室に閉じ込める。
- `Cabin` : 乗客が滞在する客室番号。形式は deck / num / side。 side は P（スターボード）か S（ポート）。
- `Destination` : 乗客が降機する惑星。
- `Age` : 年齢
- `VIP` : 乗客が航海中に特別な VIP サービス料金を支払ったかどうか。
- `RoomService` : ルームサービスでの利用金額
- `FoodCourt` : フードコートでの利用金額
- `ShoppingMall` : ショッピングモールでの利用金額
- `Spa` : スパでの利用金額
- `VRDeck` : VRDeckでの利用金額
- `Name` : 乗客の姓名（敬称ではない）

In [411]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


# 関数の定義

## LightGBM

### ハイパーパラメータの調整

- `num_leaves`
- `colsample_bytree`
- `subsample`
- `subsample_freq = 1` : `subsample` と併せて設定

In [412]:
def train_lgb(train_df, test_df, feature_cols, target_col="Transported"):
    X_train = train_df[feature_cols].copy()
    X_test = test_df[feature_cols].copy()
    
    # boolean型やカテゴリ型を考慮し、数値(0/1)に変換
    y_train = train_df[target_col].astype(int)

    # カテゴリ変数のリストを取得
    categorical_features = list(X_train.select_dtypes(include=['category', 'object']).columns)
    
    # カテゴリ変数を LightGBM が扱える category 型に明示的に変換
    for col in categorical_features:
        X_train[col] = X_train[col].astype('category')
        X_test[col] = X_test[col].astype('category')

    # 返り値用の配列を初期化
    oof_train = np.zeros(len(train_df))
    y_preds = []
    models = []

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    callbacks = [lgb.early_stopping(stopping_rounds=10, verbose=False)]
    
    params = {
        "objective": "binary",
        "metric": "binary_logloss",
        "num_leaves": 20,
        "colsample_bytree": 0.9,  # 列（特徴量）の 90% を使用
        "subsample": 0.9,  # 行（データ）の 90% を使用
        "subsample_freq": 1,  # 木を1本作るごとにサンプリングを更新
        "verbosity": -1,
        "random_state": 0
    }

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X_train, y_train)):
        X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
        X_va, y_va = X_train.iloc[valid_idx], y_train.iloc[valid_idx]

        lgb_train = lgb.Dataset(X_tr, y_tr, categorical_feature=categorical_features)
        lgb_eval = lgb.Dataset(X_va, y_va, reference=lgb_train, categorical_feature=categorical_features)

        model = lgb.train(
            params,
            lgb_train,
            valid_sets=[lgb_train, lgb_eval],
            num_boost_round=1000,
            callbacks=callbacks
        )

        # OOFの予測値を代入
        oof_train[valid_idx] = model.predict(X_va)

        # テストデータの予測
        y_pred = model.predict(X_test)
        y_preds.append(y_pred)
        models.append(model)

    return oof_train, y_preds, models

## CatBoost

In [413]:
from catboost import CatBoostClassifier, Pool


def train_cat(train_df, test_df, feature_cols, target_col="Transported"):
    X_train = train_df[feature_cols].copy()
    X_test = test_df[feature_cols].copy()
    y_train = train_df[target_col].astype(int)

    # カテゴリ変数のリストを取得（文字列型や category 型の列）
    cat_features = list(
        X_train.select_dtypes(include=["category", "object"]).columns
    )

    # 欠損値を文字列 'missing' で埋めておく（CatBoost の処理用）
    for col in cat_features:
        X_train[col] = X_train[col].astype(str).fillna("missing")
        X_test[col] = X_test[col].astype(str).fillna("missing")

    oof_train = np.zeros(len(train_df))
    y_preds = []
    models = []

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

    for fold, (train_idx, valid_idx) in enumerate(
        cv.split(X_train, y_train)
    ):
        X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
        X_va, y_va = X_train.iloc[valid_idx], y_train.iloc[valid_idx]

        train_pool = Pool(X_tr, y_tr, cat_features=cat_features)
        valid_pool = Pool(X_va, y_va, cat_features=cat_features)
        test_pool = Pool(X_test, cat_features=cat_features)

        model = CatBoostClassifier(
            iterations=1500,  # 決定木の最大数（デフォルト 500 ）
            learning_rate=0.03,
            depth=4,
            eval_metric="Logloss",
            random_seed=0,
            verbose=False,
            early_stopping_rounds=20,
        )

        model.fit(train_pool, eval_set=valid_pool)

        # 予測確率を取得
        oof_train[valid_idx] = model.predict_proba(valid_pool)[:, 1]
        y_pred = model.predict_proba(test_pool)[:, 1]

        y_preds.append(y_pred)
        models.append(model)

    return oof_train, y_preds, models

## XGBoost

In [414]:
import xgboost as xgb


def train_xgb(train_df, test_df, feature_cols, target_col="Transported"):
    X_train = train_df[feature_cols].copy()
    X_test = test_df[feature_cols].copy()
    y_train = train_df[target_col].astype(int)

    # 1. カテゴリ対象列を取得
    cat_features = list(
        X_train.select_dtypes(
            include=["category", "object", "bool"]
        ).columns
    )

    # 2. 全て一旦文字列 (str) に変換してから category 型にする（エラー回避）
    for col in cat_features:
        X_train[col] = X_train[col].astype(str).astype("category")
        X_test[col] = X_test[col].astype(str).astype("category")

    oof_train = np.zeros(len(train_df))
    y_preds = []
    models = []

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

    params = {
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "learning_rate": 0.03,
        "max_depth": 4,
        "colsample_bytree": 0.8,
        "subsample": 0.8,
        "tree_method": "hist",
        "enable_categorical": True,
        "random_state": 0,
    }

    for fold, (train_idx, valid_idx) in enumerate(
        cv.split(X_train, y_train)
    ):
        X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
        X_va, y_va = X_train.iloc[valid_idx], y_train.iloc[valid_idx]

        dtrain = xgb.DMatrix(X_tr, label=y_tr, enable_categorical=True)
        dvalid = xgb.DMatrix(X_va, label=y_va, enable_categorical=True)
        dtest = xgb.DMatrix(X_test, enable_categorical=True)

        model = xgb.train(
            params,
            dtrain,
            num_boost_round=1000,
            evals=[(dtrain, "train"), (dvalid, "valid")],
            early_stopping_rounds=20,
            verbose_eval=False,
        )

        oof_train[valid_idx] = model.predict(dvalid)
        y_preds.append(model.predict(dtest))
        models.append(model)

    return oof_train, y_preds, models

# EDA

## グループ ID

`PassengerId` から、グループ ID を抽出 ( `group_id` ) 。

In [415]:
# PassengerId からグループ ID (例: "0001") を抽出
train_group_id = train["PassengerId"].apply(lambda x: x.split("_")[0])
test_group_id = test["PassengerId"].apply(lambda x: x.split("_")[0])

train["group_id"] = train_group_id
test["group_id"] = test_group_id

## `Age` のカテゴリ化

In [416]:
# 年齢層の作成
def get_age_group(age):
    if pd.isna(age): return np.nan
    if age < 13: return 'child'        # 0〜12歳（子供）
    elif age < 18: return 'teen'       # 13〜17歳（ティーン）
    elif age < 35: return 'young'      # 18〜34歳（若手）
    elif age < 60: return 'adult'      # 35〜59歳（大人）
    else: return 'senior'             # 60歳以上（シニア）

train['age_group'] = train['Age'].apply(get_age_group)
test['age_group'] = test['Age'].apply(get_age_group)

## 家族構成の特徴量を作成

In [417]:
# --- グループごとの年齢統計量の作成 ---
# 各グループ内の最小年齢・最大年齢・人数を集計
group_age_stats = train.groupby("group_id")["Age"].agg(group_min_age="min", group_max_age="max")

# train / test にマージ
train = train.merge(group_age_stats, on="group_id", how="left")
test = test.merge(group_age_stats, on="group_id", how="left")

# 1. グループ内の年齢差（最高齢 - 最年少）
# 年齢差が大きい = 親子や三世代旅行の可能性が高い
train["group_age_diff"] = train["group_max_age"] - train["group_min_age"]
test["group_age_diff"] = test["group_max_age"] - test["group_min_age"]

# 2. 単身（一人旅）フラグ
# グループ人数が 1、またはグループ内の年齢差がない（0歳）
train["is_alone"] = (train["group_age_diff"].isna() | (train["group_age_diff"] == 0)).astype(int)
test["is_alone"] = (test["group_age_diff"].isna() | (test["group_age_diff"] == 0)).astype(int)

## 不採用案

- `Name` から姓を抽出し、家族関係を特定。

- 出費関連特徴量の対数変換、全く使われない。

# 欠損値の補完

## `Cabin`

In [418]:
# 1. Cabin 列を / で分解して deck, num, side を作成
train[['deck', 'num', 'side']] = train['Cabin'].str.split('/', expand=True)
test[['deck', 'num', 'side']] = test['Cabin'].str.split('/', expand=True)

# 3. train データから「グループごとの最頻値」マップを作成
# （複数ある場合は最初の最頻値を採用）
group_deck_map = train.groupby('group_id')['deck'].agg(lambda x: x.mode()[0] if not x.mode().empty else np.nan)
group_side_map = train.groupby('group_id')['side'].agg(lambda x: x.mode()[0] if not x.mode().empty else np.nan)

# 4. まずグループ内の他メンバーの値で補完
train['deck'] = train['deck'].fillna(train['group_id'].map(group_deck_map))
test['deck'] = test['deck'].fillna(test['group_id'].map(group_deck_map))

train['side'] = train['side'].fillna(train['group_id'].map(group_side_map))
test['side'] = test['side'].fillna(test['group_id'].map(group_side_map))

# 5. それでも残った欠損値（グループ内にデータがない場合）を train 全体の最頻値で補完
deck_mode = train['deck'].mode()[0]
side_mode = train['side'].mode()[0]

train['deck'] = train['deck'].fillna(deck_mode)
test['deck'] = test['deck'].fillna(deck_mode)

train['side'] = train['side'].fillna(side_mode)
test['side'] = test['side'].fillna(side_mode)

## `HomePlane`, `Destination`

In [419]:
# --- HomePlanet のグループ内補完 ---
# 1. train からグループごとの HomePlanet の最頻値マップを作成
group_homeplanet_map = train.groupby("group_id")["HomePlanet"].agg(
    lambda x: x.mode()[0] if not x.mode().empty else np.nan
)

# 2. グループ内の他メンバーの値で補完
train["HomePlanet"] = train["HomePlanet"].fillna(train["group_id"].map(group_homeplanet_map))
test["HomePlanet"] = test["HomePlanet"].fillna(test["group_id"].map(group_homeplanet_map))

# 3. グループ内でも決まらなかった欠損値を train 全体の最頻値で補完
homeplanet_mode = train["HomePlanet"].mode()[0]
train["HomePlanet"] = train["HomePlanet"].fillna(homeplanet_mode)
test["HomePlanet"] = test["HomePlanet"].fillna(homeplanet_mode)


# --- Destination のグループ内補完 ---
# 1. train からグループごとの Destination の最頻値マップを作成
group_destination_map = train.groupby("group_id")["Destination"].agg(
    lambda x: x.mode()[0] if not x.mode().empty else np.nan
)

# 2. グループ内の他メンバーの値で補完
train["Destination"] = train["Destination"].fillna(train["group_id"].map(group_destination_map))
test["Destination"] = test["Destination"].fillna(test["group_id"].map(group_destination_map))

# 3. グループ内でも決まらなかった欠損値を train 全体の最頻値で補完
destination_mode = train["Destination"].mode()[0]
train["Destination"] = train["Destination"].fillna(destination_mode)
test["Destination"] = test["Destination"].fillna(destination_mode)

## `CryoSleep`, 出費関連

In [420]:
# 出費系の列リスト
spending_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

# --- 1. 出費合計の仮計算 ---
# 欠損値を無視して合計（すべての出費が NaN の人は 0.0 になるため、後で別途処理）
train["total_spending"] = train[spending_cols].sum(axis=1)
test["total_spending"] = test[spending_cols].sum(axis=1)

# --- 2. 出費がある人の CryoSleep 欠損を False で補完 ---
# お金を使っている人は絶対に冷凍睡眠（True）ではない
train.loc[
    (train["CryoSleep"].isna()) & (train["total_spending"] > 0), "CryoSleep"
] = False
test.loc[
    (test["CryoSleep"].isna()) & (test["total_spending"] > 0), "CryoSleep"
] = False

# --- 3. CryoSleep == True の人の出費欠損を 0 で補完 ---
# 冷凍睡眠中の人は施設利用不可なので出費は 0
for col in spending_cols:
    train.loc[(train["CryoSleep"] == True) & (train[col].isna()), col] = 0.0
    test.loc[(test["CryoSleep"] == True) & (test[col].isna()), col] = 0.0

# --- 4. 子供（age_group == 'child'）の出費欠損を 0 で補完 ---
# 年齢層が child の乗客は施設利用不可（出費 0）として補完
for col in spending_cols:
    train.loc[(train["age_group"] == "child") & (train[col].isna()), col] = 0.0
    test.loc[(test["age_group"] == "child") & (test[col].isna()), col] = 0.0

# --- 5. 補完後の値で total_spending を更新 ---
train["total_spending"] = train[spending_cols].sum(axis=1)
test["total_spending"] = test[spending_cols].sum(axis=1)

# 実行

## LightGBM 単体

In [421]:
# 1. 特徴量の選定
ignore_cols = ["PassengerId", "Name", "Transported", "Cabin", "num", "group_id"]
feature_cols = [c for c in train.columns if c not in ignore_cols]

# 2. 数値型の欠損値補完（LightGBMのエラー防止用）
num_cols = train[feature_cols].select_dtypes(include=["float64", "int64"]).columns
for col in num_cols:
    train[col] = train[col].fillna(train[col].median())
    test[col] = test[col].fillna(train[col].median())

# bool 型の欠損値を False で補完（または型変換エラーを防止）
bool_cols = train[feature_cols].select_dtypes(include=["bool"]).columns
for col in bool_cols:
    train[col] = train[col].fillna(False).astype(str)
    test[col] = test[col].fillna(False).astype(str)

# 3. 関数の実行（文字列やboolの欠損値・型変換は関数内の処理にまかせる）
# oof_train, y_preds, models = train_lgb(
#     train_df=train,
#     test_df=test,
#     feature_cols=feature_cols,
#     target_col="Transported",
# )

In [422]:
# OOF (Out-Of-Fold) 予測に基づくモデルの評価
# Accuracy（正解率）の計算: 予測確率が 0.51 を超えるなら 1、以下なら 0
# oof_preds_binary = oof_train > 0.51
# oof_accuracy = accuracy_score(train["Transported"], oof_preds_binary)

# Log Loss（対数損失）の計算
# oof_logloss = log_loss(train["Transported"], oof_train)

# print("【前回】")
# print(f"CV Accuracy = 0.8094")
# print(f"CV Log Loss = 0.3994")
# print("----------------------------------------")
# print("【今回】")
# print(f"CV Accuracy = {oof_accuracy:.4f}")
# print(f"CV Log Loss = {oof_logloss:.4f}")

# 特徴量重要度

- Split : モデルが特徴量を使ってデータを何回「分割」したか。単純な指標。
- Gain : 特徴量によって「損失（誤差）」がどれだけ減少したか。特徴量の寄与度を測るのに適している。

In [423]:
# def plot_feature_importance_gain(models, feature_cols, top_n=20):
    
    # 全 Fold の Gain をまとめて平均
    # fi_matrix = [m.feature_importance(importance_type="gain") for m in models]
    # fi_df = (
    #     pd.DataFrame(fi_matrix, columns=feature_cols)
    #     .mean()
    #     .reset_index()
    # )
    # fi_df.columns = ["feature", "importance_mean"]
    # fi_df = fi_df.sort_values(by="importance_mean", ascending=False).head(top_n)

    # 描画
    # plt.figure(figsize=(12, 8))
    # sns.barplot(
    #     x="importance_mean",
    #     y="feature",
    #     data=fi_df,
    #     hue="feature",
    #     legend=False,
    #     palette="viridis",
    # )
    # plt.title(f"Feature Importance (Gain) - Top {top_n}", fontsize=16)
    # plt.xlabel("Importance (Mean Gain)", fontsize=14)
    # plt.ylabel("Feature", fontsize=14)
    # plt.show()


# --- Plot ---
# plot_feature_importance_gain(models, feature_cols, top_n=20)

# アンサンブル

In [424]:
# --- 1. 3つのモデルの学習 ---
oof_lgb, y_preds_lgb, models_lgb = train_lgb(
    train_df=train,
    test_df=test,
    feature_cols=feature_cols,
    target_col="Transported",
)

oof_cat, y_preds_cat, models_cat = train_cat(
    train_df=train,
    test_df=test,
    feature_cols=feature_cols,
    target_col="Transported",
)

oof_xgb, y_preds_xgb, models_xgb = train_xgb(
    train_df=train,
    test_df=test,
    feature_cols=feature_cols,
    target_col="Transported",
)

# --- 2. 3モデルのアンサンブル（比率 0.5 : 0.3 : 0.2） ---
w_lgb, w_cat, w_xgb = 0.5, 0.3, 0.2

oof_ens = w_lgb * oof_lgb + w_cat * oof_cat + w_xgb * oof_xgb

y_preds_lgb_mean = np.mean(y_preds_lgb, axis=0)
y_preds_cat_mean = np.mean(y_preds_cat, axis=0)
y_preds_xgb_mean = np.mean(y_preds_xgb, axis=0)

y_preds_ens_mean = (
    w_lgb * y_preds_lgb_mean + w_cat * y_preds_cat_mean + w_xgb * y_preds_xgb_mean
)

# --- 3. アンサンブル OOF に対する最適閾値の自動探索 ---
best_th = 0.5
best_acc = 0.0

for th in np.arange(0.30, 0.71, 0.01):
    acc = accuracy_score(train["Transported"], oof_ens > th)
    if acc > best_acc:
        best_acc = acc
        best_th = th

# --- 4. 評価の表示 ---
acc_lgb = accuracy_score(train["Transported"], oof_lgb > best_th)
acc_cat = accuracy_score(train["Transported"], oof_cat > best_th)
acc_xgb = accuracy_score(train["Transported"], oof_xgb > best_th)

print(f"LGBM  単体 CV Accuracy : {acc_lgb:.4f}")
print(f"CatB  単体 CV Accuracy : {acc_cat:.4f}")
print(f"XGB   単体 CV Accuracy : {acc_xgb:.4f}")
print(f"----------------------------------------")
print(f"★ 最適閾値 (Best Threshold) : {best_th:.2f}")
print(f"★ 3-Model Ensemble CV Accuracy: {best_acc:.4f}")

# --- 5. submission.csv の作成 ---
sample_submission["Transported"] = y_preds_ens_mean > best_th
sample_submission.to_csv("submission.csv", index=False)
print("\n3モデルアンサンブル版 submission.csv の作成が完了しました。")

/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [12:00:53] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "enable_categorical" } are not used.

  self.starting_round = model.num_boosted_rounds()
/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [12:00:54] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "enable_categorical" } are not used.

  self.starting_round = model.num_boosted_rounds()
/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [12:00:55] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "enable_categorical" } are not used.

  self.starting_round = model.num_boosted_rounds()
/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [12:00:55] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "enable_categorical" } are not used.

  self.starting_round = model.num_boosted_rounds()
/usr/local/lib/python3.12/di

LGBM  単体 CV Accuracy : 0.8094
CatB  単体 CV Accuracy : 0.8059
XGB   単体 CV Accuracy : 0.8036
----------------------------------------
★ 最適閾値 (Best Threshold) : 0.50
★ 3-Model Ensemble CV Accuracy: 0.8096

3モデルアンサンブル版 submission.csv の作成が完了しました。


# 提出

In [425]:
# y_preds_mean = np.mean(y_preds, axis=0)
# sample_submission["Transported"] = y_preds_mean > 0.51
# sample_submission.to_csv("submission.csv", index=False)
# print("submission.csv の作成が完了しました。")

In [426]:
# --- 1. OOF 予測から最も Accuracy が高くなる「最適閾値」を探索 ---
# best_threshold = 0.5
# best_accuracy = 0.0

# 0.30 から 0.70 まで 0.01 刻みでループ検証
# thresholds = np.arange(0.30, 0.71, 0.01)
# for th in thresholds:
#     acc = accuracy_score(train["Transported"], oof_train > th)
#     if acc > best_accuracy:
#         best_accuracy = acc
#         best_threshold = th

# print(f"★ 最適な閾値 (Best Threshold): {best_threshold:.2f}")
# print(f"★ 閾値調整後の CV Accuracy  : {best_accuracy:.4f}")

# --- 2. テストデータの予測確率の平均を計算 ---
# y_preds_mean = np.mean(y_preds, axis=0)

# --- 3. 最適閾値を使って二値化し、submission.csv を作成 ---
# sample_submission["Transported"] = y_preds_mean > best_threshold
# sample_submission.to_csv("submission.csv", index=False)

# print("\n最適閾値を適用した submission.csv の作成が完了しました。")